# D1-05 Uncertainty and Monte Carlo basics

This notebook introduces a practitioner-facing Monte Carlo workflow in `brightway` using uncertainty information already present in the imported BAFU database.

We focus on how uncertainty distributions propagate into LCIA scores and how to interpret overlapping score distributions.


## Learning goals

- Find uncertainty information attached to exchanges in a real database.
- Run Monte Carlo simulations with `brightway`.
- Summarize score distributions with means, percentiles, and histograms.
- Compare two activities and discuss whether the ranking is robust.


## Background reference

- Mutel, C. (2017). *Brightway: An open source framework for life cycle assessment*. Journal of Open Source Software, 2(12), 236. https://doi.org/10.21105/joss.00236


## 1) Load the imported BAFU database and choose two comparison activities

For this notebook we compare two combustion-related activities found by search terms.
If your search results differ slightly, that is acceptable as long as both activities are meaningful and non-empty.


In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display
import bw2calc as bc
import bw2data as bd

pd.options.display.float_format = '{:,.3f}'.format

bd.projects.set_current('barcelona-rlcia-2026')
database_name = 'bafu-2025'

if database_name not in bd.databases:
    print("Run D1-04 first so that 'bafu-2025' exists in the current project.")
else:
    db = bd.Database(database_name)

    def first_hit(queries):
        for query in queries:
            hits = db.search(query, limit=5)
            if hits:
                return hits[0]
        return None

    coal_activity = first_hit(['hard coal', 'coal', 'furnace'])
    gas_activity = first_hit(['natural gas, burned', 'natural gas', 'gas'])
    climate_methods = [m for m in bd.methods if 'climate change' in ' | '.join(m).lower()]
    method = climate_methods[0] if climate_methods else next(iter(bd.methods))

    print('Coal-like activity:', coal_activity['name'] if coal_activity else 'NOT FOUND')
    print('Gas-like activity:', gas_activity['name'] if gas_activity else 'NOT FOUND')
    print('Method used:', method)


## 2) Inspect uncertainty information on exchanges

Brightway stores uncertainty information directly on exchanges.
The conventions for `uncertainty type`, `loc`, `scale`, and bounds follow the `stats_arrays` data model used throughout Brightway.


In [ ]:
def show_uncertain_exchanges(activity, limit=5):
    print(f"\nActivity: {activity['name']}")
    shown = 0
    for exc in activity.exchanges():
        data = exc.as_dict()
        if data.get('uncertainty type') not in (None, 0):
            print({
                'input': exc.input['name'],
                'type': data.get('type'),
                'amount': data.get('amount'),
                'uncertainty type': data.get('uncertainty type'),
                'loc': data.get('loc'),
                'scale': data.get('scale'),
                'minimum': data.get('minimum'),
                'maximum': data.get('maximum'),
            })
            shown += 1
            if shown >= limit:
                break
    if shown == 0:
        print('No uncertain exchanges found in the first scan.')

if database_name in bd.databases and coal_activity is not None and gas_activity is not None:
    show_uncertain_exchanges(coal_activity, limit=3)
    show_uncertain_exchanges(gas_activity, limit=3)


## Checkpoint 1

Pick one exchange with uncertainty information from either comparison activity and print its `amount`, `uncertainty type`, `loc`, and `scale`.


In [ ]:
# TODO
# candidate = ...
# data = candidate.as_dict()
# print(...)


In [ ]:
if database_name not in bd.databases or coal_activity is None:
    print('Run the previous cells first.')
else:
    candidate = next(exc for exc in coal_activity.exchanges() if exc.as_dict().get('uncertainty type') not in (None, 0))
    data = candidate.as_dict()
    print({
        'input': candidate.input['name'],
        'amount': data.get('amount'),
        'uncertainty type': data.get('uncertainty type'),
        'loc': data.get('loc'),
        'scale': data.get('scale'),
    })


## 3) Run Monte Carlo simulations

We now sample uncertain exchange values repeatedly and observe the resulting score distributions.


In [ ]:
iterations = 300

def mc_scores(activity, method, iterations=300):
    mc = bc.MonteCarloLCA({activity: 1}, method)
    return np.array([next(mc) for _ in range(iterations)], dtype=float)

if database_name not in bd.databases or coal_activity is None or gas_activity is None:
    print('Activity selection is incomplete. Check the search results above.')
else:
    coal_scores = mc_scores(coal_activity, method, iterations=iterations)
    gas_scores = mc_scores(gas_activity, method, iterations=iterations)

    summary = pd.DataFrame([
        {
            'activity': coal_activity['name'],
            'mean': coal_scores.mean(),
            'std': coal_scores.std(ddof=1),
            'p05': np.percentile(coal_scores, 5),
            'p50': np.percentile(coal_scores, 50),
            'p95': np.percentile(coal_scores, 95),
        },
        {
            'activity': gas_activity['name'],
            'mean': gas_scores.mean(),
            'std': gas_scores.std(ddof=1),
            'p05': np.percentile(gas_scores, 5),
            'p50': np.percentile(gas_scores, 50),
            'p95': np.percentile(gas_scores, 95),
        },
    ])

    display(summary)


In [ ]:
if 'coal_scores' in globals() and 'gas_scores' in globals():
    plt.figure(figsize=(8, 4))
    plt.hist(coal_scores, bins=30, alpha=0.6, label='Coal-like activity')
    plt.hist(gas_scores, bins=30, alpha=0.6, label='Gas-like activity')
    plt.xlabel('Impact score')
    plt.ylabel('Frequency')
    plt.title('Monte Carlo score distributions')
    plt.legend()
    plt.tight_layout()
    plt.show()


## 4) Percentiles and overlap

A practical question is whether the uncertainty ranges overlap so much that a simple ranking becomes weak or unreliable.


In [ ]:
if 'coal_scores' in globals() and 'gas_scores' in globals():
    interval_table = pd.DataFrame([
        {
            'activity': 'coal-like',
            'p05': np.percentile(coal_scores, 5),
            'p50': np.percentile(coal_scores, 50),
            'p95': np.percentile(coal_scores, 95),
        },
        {
            'activity': 'gas-like',
            'p05': np.percentile(gas_scores, 5),
            'p50': np.percentile(gas_scores, 50),
            'p95': np.percentile(gas_scores, 95),
        },
    ])
    overlap = not (interval_table.loc[0, 'p95'] < interval_table.loc[1, 'p05'] or interval_table.loc[1, 'p95'] < interval_table.loc[0, 'p05'])
    display(interval_table)
    print('Do the 5th-95th percentile intervals overlap?', overlap)


## Checkpoint 2

Use the summary table and histogram to decide whether the ranking between the two activities is robust.
If the intervals overlap strongly, explain why extra care is needed before claiming one option is clearly better.


In [ ]:
# TODO
# Write 2-3 sentences here as markdown or comments.


In [ ]:
if 'coal_scores' in globals() and 'gas_scores' in globals():
    coal_p05, coal_p95 = np.percentile(coal_scores, [5, 95])
    gas_p05, gas_p95 = np.percentile(gas_scores, [5, 95])
    overlap = not (coal_p95 < gas_p05 or gas_p95 < coal_p05)
    print('Interpretation:')
    if overlap:
        print('- The intervals overlap, so a simple ranking is not fully robust.')
        print('- A stronger claim would need additional analysis or a larger difference in distributions.')
    else:
        print('- The intervals are separated, so the ranking looks more robust in this comparison.')
    print('- In D1-07, contribution analysis helps us ask which parts of the model drive the score.')


## Recap

After this notebook, you should now be able to:

- identify uncertainty information on exchanges
- run Monte Carlo simulations in `brightway`
- summarize score distributions with percentiles and histograms
- explain why overlapping uncertainty ranges weaken simple rankings
